# Day 15

---


In [ ]:
import grama as gr
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
DF = gr.Intention()

# Set figure options
plt.rcParams['figure.figsize'] = [6, 6] # Need square aspect ratio for gradients to appear normal
plt.rcParams['figure.dpi'] = 100 # 200 e.g. is really fine, but slower


## Probability-related functions

---


In [ ]:
# First, define a distribution
mg_test = gr.marg_mom("norm", mean=0, sd=1)
mg_test

In [ ]:
# Evaluate the probability density function
mg_test.d(0)

In [ ]:
# Evaluate the cumulative distribution function (CDF)
mg_test.p(0)

In [ ]:
# Evaluate the quantile
mg_test.q(0)
# mg_test.q(0.25)

In [ ]:
# Plot the PDF, for ease of understanding
(
    gr.df_make(x=gr.linspace(-3, +3, 100))
    >> gr.tf_mutate(d=mg_test.d(DF.x))
    >> gr.ggplot(gr.aes("x", "d"))
    + gr.geom_line()
)

## Distributions and Summaries

Create distributions by name

- See Table D.6.1 of AMAU for a shortlist of distribution names
- All distributions in [scipy](https://docs.scipy.org/doc/scipy/reference/stats.html) are supported

In [ ]:
# Create distributions by specifying shape and moments
mg_unif = gr.marg_mom("uniform", lo=0, up=1)
mg_unif

In [ ]:
# mg_beta = gr.marg_mom("beta", lo=0, up=1) # Not enough information
mg_beta = gr.marg_mom("beta", lo=0, up=1, mean = 0.30, sd=0.25)
mg_beta

In [ ]:
# Plot the PDF, for ease of understanding
(
    gr.df_make(x=gr.linspace(0, 1, 100))
    >> gr.tf_mutate(d=mg_beta.d(DF.x))
    >> gr.ggplot(gr.aes("x", "d"))
    + gr.geom_line()
)

In [ ]:
mg_norm = gr.marg_mom("norm", mean=0, sd=1)
mg_gnorm = gr.marg_mom("gennorm", mean=0, sd=1, kurt=4)

# Plot the PDF, for ease of understanding
(
    gr.df_make(x=gr.linspace(-5, +5, 100))
    >> gr.tf_mutate(d_norm=mg_norm.d(DF.x))
    >> gr.tf_mutate(d_gennorm=mg_gnorm.d(DF.x))
    >> gr.tf_pivot_longer(
        columns=["d_norm", "d_gennorm"],
        names_pattern=r"(d)_(\w+)",
        names_to=[".value", "dist"],
    )
    
    >> gr.ggplot(gr.aes("x", "d"))
    + gr.geom_line(gr.aes(linetype="dist"))
    # + gr.scale_y_log10()
)

## Modeling with distributions

Two approaches:

1. Fit to data
2. Fit using moments

### 1. Fit to data

If we're lucky enough to have a dataset with repeated, independent observations, we can *fit* a distribution. First, we inspect the data to get a sense of shape:

In [ ]:
from grama.data import df_shewhart

(
    df_shewhart
    >> gr.ggplot(gr.aes("tensile_strength"))
    + gr.geom_histogram(bins=20)
)

We can use `gr.marg_fit()` to fit a named distribution to a dataset. We should select candidates that are physically plausible.

In [ ]:
# Bounded values
mg_tys_beta = gr.marg_fit("beta", df_shewhart.tensile_strength)
# Bounded below
mg_tys_lognorm = gr.marg_fit("lognorm", df_shewhart.tensile_strength)

(
    gr.df_make(tys=gr.linspace(20000, 45000))
    >> gr.tf_mutate(
        d_beta=mg_tys_beta.d(DF.tys),
        d_lognorm=mg_tys_lognorm.d(DF.tys),
    )
    >> gr.tf_pivot_longer(
        columns=["d_beta", "d_lognorm"],
        names_pattern=r"(d)_(\w+)",
        names_to=[".value", "dist"],
    )
    
    >> gr.ggplot(gr.aes("tys", "d"))
    + gr.geom_histogram(
        data=df_shewhart,
        mapping=gr.aes("tensile_strength", y=gr.after_stat("density")),
        bins=20,
    )
    + gr.geom_line(gr.aes(linetype="dist"))
)

A more sensitive tool for assessing distributions is the *QQ Plot*, which we'll discuss later (see AMAU Appendix B.4):

In [ ]:
(
    df_shewhart
    >> gr.tf_mutate(
        q=gr.qqvals(DF.tensile_strength, "beta")
    )

    >> gr.ggplot(gr.aes("q", "tensile_strength"))
    + gr.geom_abline(slope=1, intercept=0)
    + gr.geom_point()
    + gr.labs(
        x="Theoretical Quantiles",
        y="Observed Quantiles\n(Tensile Strength)",
        title="Beta Fit",
    )
)

We can see the weird "pileup" of observations just below 35000.

### 2. Fit using moments

In many cases, we don't have data with multiple observations. We may have to settle for specifying the distribution in terms of moments.

| Material  | Property | Mean | CoV |
|-----------|----------|------|-----|
| Ti 6Al 4V | Strength (ksi) | 160 | 0.06 |

In [ ]:
mg_ti64 = gr.marg_mom("norm", mean=160, cov=0.06)
mg_ti64

Problem: For other distributions, we may need higher moments

In [ ]:
mg_ti64_lognorm = gr.marg_mom("lognorm", mean=160, cov=0.06)
# mg_ti64_lognorm = gr.marg_mom("lognorm", mean=160, cov=0.06, skew=0) # Not all moments are feasible
# mg_ti64_lognorm = gr.marg_mom("lognorm", mean=160, cov=0.06, skew=0.1) # ???
mg_ti64_lognorm